# Text als Daten

Dieser Kurs verwendet Schlagzeilen und Teaser zweier Tageszeitungen aus demselben Jahr - **Der Standard** aus Wien und die **Süddeutsche Zeitung** aus München, Juni 2015 bis Mai 2016 -, um das Standardwerkzeug des Text Mining durchzuarbeiten, vom Zählen von Wörtern bis zum Trainieren von Klassifikatoren. Sechs Skripte lang bleiben wir bewusst bei **klassischen, nachvollziehbaren Methoden**; ein großes Sprachmodell (LLM) kommt erst im letzten Skript vor, wo es mit allem verglichen werden kann, was wir davor aufgebaut haben.

Am Ende dieses Skripts kannst du:

- entscheiden, was als **Dokument** zählt, und verstehen, warum das die erste und folgenreichste Entscheidung jeder Textanalyse ist;
- deutschen Text mit einem regulären Ausdruck **tokenisieren** und erklären, wo naive Ansätze scheitern;
- **Normalisierung** und **Stoppwort-Entfernung** anwenden und wissen, was dabei jeweils verloren geht;
- ein Korpus mit **Häufigkeitsverteilungen**, dem Zipfschen Gesetz und N-Grammen beschreiben.

Alles in diesem Skript verwendet nur die Standardbibliothek, pandas und matplotlib (dazu die Stoppwortliste von NLTK). Ein Sprachmodell kommt noch nicht vor - das folgt in Skript 02 -, daher ist jeder Schritt hier sichtbar, statt in einem Bibliotheksaufruf zu verschwinden.

## Voraussetzungen

In [ ]:
# Zeige alle installierten Pakete mit ihren Versionen
%pip list

Die Liste zeigt alle Pakete der aktiven Python-Umgebung mit ihren Versionen. Prüfe, ob die oben genannten Pakete dabei sind (etwa `spacy`, `scikit-learn`, `sentence-transformers`, `nltk`) und ob die drei spaCy-Modelle `de-core-news-sm`, `-md` und `-lg` installiert sind. Der Hinweis am Ende stammt von Jupyter und betrifft nur Installationen aus dem Notebook heraus.

## Die Daten

Beide Zeitungen stammen aus frei verfügbaren Forschungskorpora:

- **Der Standard** aus dem *One Million Posts Corpus* des Österreichischen Forschungsinstituts für Artificial Intelligence (OFAI; Schabus et al. 2017). Das Korpus wurde für die Analyse von Leserkommentaren gebaut und enthält dazu die 12.087 Artikel, unter denen kommentiert wurde - mit Titel, Erscheinungszeitpunkt, Ressort und HTML-Text. Lizenz: CC BY-NC-SA 4.0.
- **Süddeutsche Zeitung** aus *MLSUM*, einem mehrsprachigen Korpus für automatische Zusammenfassung (Scialom et al. 2020). Sein deutscher Teil besteht aus Artikeln von sueddeutsche.de mit Titel, Text, von der Redaktion geschriebener Zusammenfassung (dem Teaser), Ressort und Monat. Wir behalten die Artikel aus denselben zwölf Monaten. Lizenz: nur für nicht-kommerzielle Forschung und Lehre.

Das Hilfsmodul `textmining.py` lädt beide Quellen beim ersten Aufruf herunter (rund 1 GB, einige Minuten) und speichert die Artikel als zwei Parquet-Dateien im Ordner `data/` neben den Skripten - oder, falls gesetzt, im Ordner aus der Umgebungsvariable `TEXTMINING_DATA`.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Reguläre Ausdrücke und Zähler aus der Standardbibliothek
import re
from collections import Counter

# Visualisierung und Datentransformation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Paket für die Stoppwortliste
import nltk

# Hilfsmodul des Kurses: Laden und Bereinigen des Korpus
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Von Rohdaten zu Dokumenten

`tm.load_raw()` liefert die beiden Quellen so, wie sie geliefert werden: eine Zeile je Artikel, aber noch ohne jede Entscheidung darüber, was davon ein brauchbares Dokument ist.

In [ ]:
# Lade die beiden Quellen unverändert (beim ersten Aufruf: Download und Umwandlung)
standard_raw, sz_raw = tm.load_raw()
print(f"Der Standard:        {len(standard_raw):>6,} Artikel, Spalten: {list(standard_raw.columns)}")
print(f"Süddeutsche Zeitung: {len(sz_raw):>6,} Artikel, Spalten: {list(sz_raw.columns)}")

Die beiden Quellen sehen sich nur auf den ersten Blick ähnlich. Der Standard liefert einen exakten Zeitstempel, einen Ressort-Pfad und den Artikel als HTML; die SZ liefert eine URL, das Ressort, die Zusammenfassung der Redaktion (`teaser`), den Text als reinen Text und nur den **Monat** des Erscheinens - MLSUM hat das Datum nicht genauer gespeichert. Das wird uns später begleiten: Zeitreihen über beide Zeitungen sind nur je Monat möglich.

Ein Dokument ist in diesem Kurs eine **Schlagzeile**; für Methoden, die mehr Text brauchen, gibt es zusätzlich Schlagzeile plus Teaser. Bis dahin braucht es mehrere Entscheidungen, und `tm.load_corpus()` - die gemeinsame Ladefunktion, die jedes Skript verwendet - trifft alle. Hier betrachten wir sie einzeln, damit sichtbar wird, was jede davon entfernt oder verändert.

### Entscheidung 1: Ein gemeinsamer Zeitraum

Das One Million Posts Corpus umfasst die Artikel von Juni 2015 bis Mai 2016. Einige Artikel sind aber älter:

In [ ]:
# Zähle die Standard-Artikel je Jahr und zeige die ältesten
print(standard_raw["published"].dt.year.value_counts().sort_index().to_string())
standard_raw.nsmallest(5, "published")[["published", "path", "title"]]

Die älteren Artikel sind Einzelstücke aus früheren Jahren - Reisetipps, ein Literaturtext, Community-Seiten -, die nur deshalb im Korpus sind, weil im Erhebungszeitraum darunter kommentiert wurde. Sie gehören nicht zur Berichterstattung dieses Jahres. Wir behalten bei beiden Zeitungen nur Juni 2015 bis Mai 2016.

### Entscheidung 2: Nicht alles ist Journalismus

Ein Teil der Standard-Seiten richtet sich an die eigene Community: Forenregeln, Leserfragen, Off-Topic-Foren, die besten Postings der Woche. Sie liegen im Ressort `User`. Bei der SZ sind die Ressorts `service` (Gewinnspiele) und `thema` (Themenseiten) keine Artikel.

In [ ]:
# Zeige Titel aus den Community-Seiten von Der Standard und den Service-Seiten der SZ
standard_section = standard_raw["path"].map(tm.standard_section)
sz_section = sz_raw["url"].str.extract(r"sueddeutsche\.de/([^/]+)/", expand=False)
print("Der Standard, Ressort 'User':", (standard_section == "User").sum(), "Seiten")
print(standard_raw.loc[standard_section == "User", "title"].head(8).tolist())
print("\nSZ, Ressorts 'service' und 'thema':", sz_section.isin(["service", "thema"]).sum(), "Seiten")
print(sz_raw.loc[sz_section.isin(["service", "thema"]), "title"].head(6).tolist())

Die Titel bestätigen die Entscheidung: Die Community-Seiten von Der Standard sind Leserfragen, Einladungen und Hinweise in eigener Sache, die SZ-Seiten Ratgeber-Übersichten und Gewinnspiele. Keine dieser Seiten ist eine Nachricht. Zusammen sind es nur knapp 200 Seiten, sie würden aber Wörter wie *Community*, *Preis* oder *derStandard.at* künstlich häufig machen.

### Entscheidung 3: Ein Platzhalter ist keine Schlagzeile

Bei einigen hundert SZ-Artikeln steht im Titelfeld dieselbe Zeichenkette:

In [ ]:
# Zähle die häufigsten Titel der SZ
sz_raw["title"].value_counts().head(5)

"Süddeutsche.de" ist der Name der Website, nicht die Schlagzeile. Die betroffenen Artikel haben einen echten Teaser und einen echten Text; ihre Schlagzeile hat der Crawler von MLSUM aber nicht erwischt (meist sind es Texte aus der Printausgabe). Als Schlagzeilen-Dokumente sind sie wertlos, und als Gruppe würden sie das "Wort" *Süddeutsche.de* zu einem der häufigsten Terme des Korpus machen. Wir entfernen sie.

### Entscheidung 4: Formate sind keine Meldungen

Dieselbe Liste zeigt das nächste Problem: Titel wie *Sport in der Region* oder *SZ Espresso …* benennen eine **Rubrik**, unter der jeden Tag ein anderer Inhalt erscheint. Auch Der Standard hat solche wiederkehrenden Titel. Wir behandeln jeden Titel, den eine Zeitung mindestens dreimal vergibt, als Format und entfernen ihn:

In [ ]:
# Titel, die eine Zeitung mindestens dreimal vergibt (ohne den Platzhalter aus Entscheidung 3)
repeated = pd.concat([
    standard_raw["title"].value_counts().rename("n").to_frame().assign(Zeitung="Der Standard"),
    sz_raw["title"].value_counts().rename("n").to_frame().assign(Zeitung="Süddeutsche Zeitung"),
])
repeated = repeated[(repeated["n"] >= tm.FORMAT_MIN_REPEATS) & (repeated.index != "Süddeutsche.de")]
print(f"{len(repeated)} wiederkehrende Titel mit zusammen {repeated['n'].sum()} Artikeln")
repeated.head(12)

Neben echten Rubriken (*Sport in der Region*, *Geschmackssache*, Newsletter-Formate) stehen hier Titel, die nur aus einer Dachzeile bestehen - ein Vereinsname, eine Sportart. Das sind keine Schlagzeilen, sondern Etiketten. Die Schwelle von drei Wiederholungen ist eine Ermessensentscheidung: Zwei Artikel mit gleichem Titel sind oft eine echte Doublette (nächste Entscheidung), ab drei handelt es sich fast immer um eine Rubrik.

### Entscheidung 5: Derselbe Artikel zweimal

Es bleiben Titel, die genau zweimal vorkommen: meist derselbe Text, der unter einer zweiten Adresse oder in einem zweiten Ressort erschienen ist. Würden wir jede Kopie zählen, bekämen die Wörter dieser Artikel das doppelte Gewicht aller anderen. Wir behalten die erste Fassung.

In [ ]:
# Titel, die in einer Zeitung genau zweimal vorkommen
for name, frame in [("Der Standard", standard_raw), ("Süddeutsche Zeitung", sz_raw)]:
    counts = frame["title"].value_counts()
    print(f"{name}: {(counts == 2).sum()} doppelte Titel, z. B. {counts[counts == 2].index[:3].tolist()}")

Bei Der Standard gibt es kaum doppelte Titel, bei der SZ rund hundert. Die Beispiele zeigen beide Fälle, die hinter einer Doublette stecken können: echte Kopien desselben Artikels und kurze, allgemeine Titel (*Im freien Fall*, *Leben und sterben lassen*), die die Redaktion für zwei verschiedene Texte verwendet hat. Mit `dedupe=True` behält `load_corpus()` in beiden Fällen nur die erste Fassung; bei so wenigen Fällen verändert das die Zählungen kaum.

### Entscheidung 6: Markup ist kein Text

Der Standard liefert seine Artikel als HTML. So beginnt ein unbearbeiteter Artikel:

In [ ]:
# Zeige den Anfang eines Artikels vor und nach der HTML-Bereinigung
in_window = standard_raw["published"] >= tm.START
raw = standard_raw.loc[in_window & standard_raw["body_html"].str.contains("<a href"), "body_html"].iloc[0]
print(raw[:600])
print()
print("bereinigt:", tm.clean_html(raw)[:400])

Ohne Bereinigung würde ein Tokenisierer Tags und Attribute in Wörter wie `div`, `class`, `copytext`, `itemprop` und `href` zerlegen - die in *jedem* Standard-Artikel vorkommen, sodass jeder spätere Vergleich der Zeitungen "entdecken" würde, dass Der Standard von HTML handelt. `tm.clean_html` entfernt die Tags und übersetzt HTML-Entities wie `&amp;` zurück in Zeichen; auch einige Titel enthalten solche Entities.

### Entscheidung 7: Spuren des CSV-Exports

Auch die SZ-Daten haben eine technische Eigenheit. Enthält eine Schlagzeile Anführungszeichen, wurde sie beim Export von MLSUM nach CSV-Regeln maskiert: in Anführungszeichen eingeschlossen, jedes innere Anführungszeichen verdoppelt.

In [ ]:
# SZ-Titel mit verdoppelten Anführungszeichen, vor und nach der Korrektur
quoted = sz_raw.loc[sz_raw["title"].str.contains('""'), "title"]
print(f"{len(quoted)} SZ-Titel betroffen")
for title in quoted.head(4):
    print(f"  {title:55} -> {tm.undo_csv_quotes(title)}")

Unkorrigiert würde jede zitierende SZ-Schlagzeile Anführungszeichen-Zeichenketten tragen, die es bei Der Standard nicht gibt - ein reines Artefakt, das in jedem Vergleich der beiden Zeitungen auftauchen würde.

### Entscheidung 8: Welcher Teaser?

Für die Spalte `text` brauchen wir zu jeder Schlagzeile einen kurzen Begleittext. Der naheliegende Kandidat, der erste Absatz des Artikels, funktioniert bei der SZ nicht: MLSUM hat den Text samt Bildunterschriften und Zwischenzeilen gespeichert, und so beginnt er oft mit einem Satz, der zum Foto gehört.

In [ ]:
# Vergleiche bei einigen SZ-Artikeln den Anfang des Textes mit dem Teaser der Redaktion
for _, r in sz_raw.sample(3, random_state=0).iterrows():
    print("Titel: ", r["title"])
    print("Text:  ", r["body"][:160], "...")
    print("Teaser:", r["teaser"], "\n")

Wir verwenden deshalb bei beiden Zeitungen den **Teaser, den die Redaktion selbst geschrieben hat**: bei der SZ die Zusammenfassung aus MLSUM, bei Der Standard den Vorspann, der im HTML als `<h2>` vor dem ersten Absatz steht (fehlt er, nehmen wir den ersten Absatz). Damit sind die beiden Teaser-Spalten von derselben Art: ein bis drei Sätze, die den Artikel ankündigen.

### Das Ergebnis

In [ ]:
# Lade das bereinigte Korpus und zeige einen Überblick pro Zeitung
corpus = tm.load_corpus()
print(f"{len(corpus):,} Dokumente")
corpus.groupby("outlet").agg(
    Land=("country", "first"),
    Artikel=("title", "size"),
    mit_Teaser=("lead", lambda s: (s != "").mean()),
    Ressorts=("section", "nunique"),
    erster_Monat=("month", "min"),
    letzter_Monat=("month", "max"),
).style.format({"mit_Teaser": "{:.0%}", "erster_Monat": "{:%Y-%m}", "letzter_Monat": "{:%Y-%m}"})

Die beiden Zeitungen sind **unterschiedlich groß**: Aus der SZ stammen deutlich mehr Artikel als aus Der Standard. Das ist kein Problem, solange wir Anteile statt absoluter Zahlen vergleichen; Skript 03 verwendet dafür Maße, die die Korpusgröße berücksichtigen. Wichtiger ist, *welche* Artikel in den beiden Teilen stecken, denn die Ressorts sind verschieden verteilt:

In [ ]:
# Anteil der gemeinsamen Ressortgruppen je Zeitung (ohne Gruppe: Ressorts ohne Gegenstück)
pd.crosstab(corpus["section_group"].fillna("ohne Gruppe"), corpus["outlet"], normalize="columns").round(2)

Die Ressorts beider Zeitungen heißen verschieden und sind verschieden geschnitten. `textmining.py` ordnet die Ressorts mit einem klaren Gegenstück einer gemeinsamen **Ressortgruppe** zu (Spalte `section_group`): Politik, Wirtschaft, Sport, Panorama, Digital. Die SZ-Daten enthalten auffallend viel Sport und Politik, die Standard-Daten einen größeren Anteil an Ressorts ohne Gegenstück (Kultur, Wissenschaft, Etat für Medien, Meinung, Lifestyle) und ein deutlich größeres Digital-Ressort. Wer die beiden Zeitungen vergleicht, sollte deshalb innerhalb einer Ressortgruppe vergleichen - Skript 03 tut das.

Wie lang sind die Dokumente?

In [ ]:
# Median der Wortanzahl in Schlagzeile und Teaser pro Zeitung
words = corpus.assign(
    title_words=corpus["title"].str.split().str.len(),
    lead_words=corpus["lead"].str.split().str.len(),
)
words.groupby("outlet")[["title_words", "lead_words"]].median()

Die Schlagzeilen sind in beiden Zeitungen ähnlich kurz, die Teaser der SZ sind länger als die Vorspänne von Der Standard. Vergleiche anhand der **Schlagzeilen allein** sind deshalb fairer, und diese verwenden die meisten Skripte; **Schlagzeile plus Teaser** (Spalte `text`) verwenden wir dort, wo eine Methode mehr Text je Dokument braucht.

## Tokenisierung

Die **Tokenisierung** zerlegt einen String in die Einheiten, die wir zählen wollen. Das klingt trivial, ist aber die Stelle, an der der meiste unbemerkte Schaden entsteht. Der naheliegende Ansatz trennt an Leerzeichen:

In [ ]:
# Wähle eine Schlagzeile mit Anführungszeichen, Doppelpunkt und Bindestrich
example = corpus.loc[
    corpus["title"].str.contains('"') & corpus["title"].str.contains(":") & corpus["title"].str.contains(r"\w-\w"),
    "title",
].iloc[0]
print(example)
print(example.split())

Satzzeichen bleiben an ihren Nachbarn hängen: Ein Wort mit nachfolgendem Doppelpunkt und dasselbe Wort am Satzende werden zu verschiedenen Tokens, und Anführungszeichen kleben an den Wörtern, die sie umschließen. Die Zählungen zerfallen, ohne dass ein Fehler gemeldet wird.

Ein **Tokenisierer auf Basis eines regulären Ausdrucks** beschreibt, was ein Wort *ist*, statt was Wörter trennt. Das naheliegende englische Muster ist eine Falle:

In [ ]:
# Ein naives Muster: nur die Buchstaben A bis Z
naive = re.compile(r"[A-Za-z]+")
print(naive.findall("Für Österreichs Bürger: Spritpreise über 2 Euro"))

Aus `Für` wurde `F` + `r`, `Österreichs` verlor seinen ersten Buchstaben, und es wurde kein Fehler gemeldet. Umlaute und ß sind im Deutschen gewöhnliche Buchstaben und müssen in der Zeichenklasse enthalten sein. Pythons `\w` deckt sie bereits ab (es ist Unicode-fähig), passt aber auch auf Ziffern und Unterstriche, daher schreiben wir die Klasse als `[^\W\d_]` aus: "Wortzeichen ohne Ziffern und Unterstrich".

Zwei weitere Entscheidungen, die das Deutsche betreffen:

- **Komposita mit Bindestrich bleiben ganz.** `Sachsen-Anhalt`, `SPÖ-Chef` und `Flüchtlings-Gipfel` sind jeweils ein einziges Konzept. Eine Trennung am Bindestrich würde `Sachsen` und `Anhalt` getrennt zählen und zwei unzusammenhängende Begriffe erfinden.
- **Zahlen fallen vorerst weg.** Preise, Datumsangaben und Ergebnisse (`25.000`, `5,3`, `2:0`) sind bedeutsam, als Vokabular erzeugen sie aber vor allem Rauschen. Skript 02 zeigt, wie ein Sprachmodell sie als Zahlen markieren kann, statt sie blind zu verwerfen.

In [ ]:
# Ein Token: Buchstabenfolge, optional mit Bindestrich verbundene weitere Buchstabenfolgen
TOKEN_RE = re.compile(r"[^\W\d_]+(?:-[^\W\d_]+)*")


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text)


print(tokenize("Für Österreichs Bürger: Spritpreise über 2 Euro"))
print(tokenize(example))

Der reguläre Ausdruck liefert genau die Wörter, die wir zählen wollen: Umlaute bleiben erhalten, Satzzeichen und Anführungszeichen sind verschwunden, die Zahl `2` fällt weg, und `U-Bahn` bleibt als Kompositum mit Bindestrich ein einziges Token.

## Normalisierung

Die **Normalisierung** legt fest, welche Oberflächenformen als dasselbe Wort zählen sollen. Der übliche erste Schritt, die Kleinschreibung, scheint im Deutschen teuer, weil **jedes Nomen großgeschrieben wird**: Nur die Groß- und Kleinschreibung unterscheidet *Laut* (ein Geräusch) von *laut* (nicht leise, oder "gemäß"), oder *Essen* (die Mahlzeit) von *essen* (das Verb).

Bevor wir entscheiden, sehen wir uns an, was in diesem Korpus tatsächlich zusammenfällt. Die folgende Zelle findet jedes Wort, das sowohl großgeschrieben als auch kleingeschrieben vorkommt, und ordnet die Paare danach, wie oft die seltenere Form vorkommt.

In [ ]:
# Zähle alle Tokens in den Titeln
title_tokens = Counter(t for title in corpus["title"] for t in tokenize(title))

# Finde Wörter, die groß- und kleingeschrieben vorkommen
collisions = pd.DataFrame(
    [
        (word, n, word.lower(), title_tokens[word.lower()])
        for word, n in title_tokens.items()
        if word[0].isupper() and word.lower() != word and title_tokens[word.lower()]
    ],
    columns=["capitalised", "n_cap", "lowercase", "n_lower"],
)

# Ordne nach der Häufigkeit der selteneren Form
collisions.assign(rarer=collisions[["n_cap", "n_lower"]].min(axis=1)).nlargest(12, "rarer")

Die Kollisionen sind ganz überwiegend **Großschreibung am Satzanfang**, keine Paare aus Nomen und Verb: *Die* und *die*, *Der* und *der*, *Mit* und *mit* sind dasselbe Wort, großgeschrieben, weil es eine Schlagzeile eröffnet. Würden wir die Schreibung beibehalten, würde jedes davon in zwei Wörter zerfallen. Echte bedeutungstragende Paare gibt es, sie sind im Vergleich aber selten.

Für das Zählen von **Themen** in Schlagzeilen ist Casefolding (Kleinschreibung) also die richtige Wahl, und so gehen wir vor. Wo die Wortart wirklich wichtig ist, bietet Skript 02 ein besseres Werkzeug als die Schreibung: einen Wortart-Tagger (POS-Tagger), der das Nomen *Laut* anhand des Kontexts von der Präposition *laut* unterscheiden kann.

Pythons `str.casefold()` ist dabei die richtige Wahl statt `lower()`: Es ist die gründlichere, Unicode-fähige Variante, die für den Vergleich ohne Berücksichtigung der Schreibung gedacht ist. Im Deutschen bildet sie außerdem `ß` auf `ss` ab, sodass `Straße` und die Schweizer Schreibweise `Strasse` zusammenfallen.

In [ ]:
# Vergleiche lower() und casefold()
print("Straße".lower(), "Straße".casefold(), "STRASSE".casefold())

`lower()` lässt das `ß` stehen, `casefold()` ersetzt es durch `ss`. Deshalb ergeben `Straße` und `STRASSE` nach `casefold()` dieselbe Zeichenkette, nach `lower()` aber nicht.

## Stoppwörter

**Stoppwörter** sind häufige Wörter mit wenig thematischem Gehalt: `der`, `und`, `mit`. Wenn wir sie entfernen, sehen nicht mehr alle Häufigkeitstabellen gleich aus. NLTK liefert eine deutsche Liste mit:

In [ ]:
# Lade die deutsche Stoppwortliste (und lade sie bei Bedarf herunter)
try:
    from nltk.corpus import stopwords

    GERMAN_STOPWORDS = set(stopwords.words("german"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords

    GERMAN_STOPWORDS = set(stopwords.words("german"))

print(len(GERMAN_STOPWORDS), "Stoppwörter, z. B.", sorted(GERMAN_STOPWORDS)[:12])
print("Negationen in der Liste:", sorted(GERMAN_STOPWORDS & {"nicht", "kein", "keine", "nichts", "nie"}))

Sieh dir immer an, was eine Stoppwortliste löscht. Diese hier entfernt Negationen, was für das Zählen von Themen harmlos ist und für alles, was mit Bedeutung oder Haltung zu tun hat, verheerend: *"keine Einigung"* und *"Einigung"* würden identisch.

Fügen wir die Schritte zusammen, erhalten wir die Vorverarbeitungsfunktion, die wir im Rest dieses Skripts verwenden:

In [ ]:
# Minimale Tokenlänge
MIN_LENGTH = 2


# Tokenisiere, wende Casefolding an und entferne Stoppwörter und zu kurze Tokens
def prepare(text: str) -> list[str]:
    tokens = (t.casefold() for t in tokenize(text))
    return [t for t in tokens if t not in GERMAN_STOPWORDS and len(t) >= MIN_LENGTH]


corpus["tokens"] = corpus["title"].map(prepare)
corpus[["outlet", "title", "tokens"]].sample(5, random_state=3)

Jede Zeile zeigt eine Schlagzeile und daneben die Tokens, die nach Tokenisierung, Casefolding und Stoppwort-Entfernung übrig bleiben. Funktionswörter wie *um*, *für*, *in* und *nach* sind verschwunden, Bindestrich-Komposita (*anti-pfingsten*) bleiben ganz, und `ß` ist zu `ss` geworden (*meissen*). Die Stichprobe stammt zufällig nur aus der SZ, was bei einem Anteil von fast drei Vierteln am Korpus nicht überrascht.

## Häufigkeitsverteilungen

Mit einem sauberen Token-Strom braucht das Zählen nicht mehr als `collections.Counter`.

In [ ]:
# Die n häufigsten Terme eines Teilkorpus
def top_terms(frame: pd.DataFrame, n: int = 15) -> pd.Series:
    counts = Counter(t for tokens in frame["tokens"] for t in tokens)
    return pd.Series(dict(counts.most_common(n)))


# Eine Spalte pro Zeitung
pd.DataFrame(
    {outlet: [f"{t} ({n})" for t, n in top_terms(group).items()] for outlet, group in corpus.groupby("outlet")}
)

Schon diese grobe Tabelle trennt die Zeitungen. Bei beiden steht das große Thema des Jahres - die Fluchtbewegung nach Europa - weit oben, aber das Heimatland jeder Redaktion scheint durch (`österreich`, `wien`, `wiener` gegenüber `deutschland`, `deutsche`, `merkel`). Auch die Zusammensetzung der beiden Teilkorpora ist sichtbar: Die vielen Sportartikel der SZ bringen `bayern`, `fc`, `fussball` und `bundesliga` nach oben, das große Web-Ressort von Der Standard bringt `apple`. Und mit `kommentar` steht bei der SZ eine Gattungsbezeichnung unter den häufigsten Wörtern - ein erster Hinweis auf die Dachzeilen weiter unten. Die Tabelle zeigt aber auch die Grenzen roher Zählungen: Die Terme, die *überall* häufig sind (`neue`, `mehr`), verdrängen jene, die jede Zeitung unverwechselbar machen. Skript 03 geht das mit TF-IDF und Keyness-Statistiken an.

### Zipfsches Gesetz

Worthäufigkeiten folgen in jeder natürlichen Sprache einem auffallend regelmäßigen Muster: Das *r*-häufigste Wort kommt ungefähr proportional zu *1/r* vor. In einer doppelt logarithmischen Darstellung der Häufigkeit gegen den Rang ergibt das eine Gerade mit einer Steigung nahe -1.

In [ ]:
# Häufigkeiten aller Terme, absteigend sortiert
counts = Counter(t for tokens in corpus["tokens"] for t in tokens)  # wird im Abschnitt "Das Problem der Komposita" wiederverwendet
freqs = np.array(sorted(counts.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)

# Passe eine Gerade im doppelt logarithmischen Raum an
slope, intercept = np.polyfit(np.log(ranks), np.log(freqs), 1)

# Zeichne Häufigkeit gegen Rang
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(ranks, freqs, ".", ms=3, color="#2F6690", alpha=0.6, label="beobachtet")
ax.loglog(ranks, np.exp(intercept) * ranks**slope, color="#B33A3A", label=f"Anpassung, Steigung {slope:.2f}")
ax.set(xlabel="Rang", ylabel="Häufigkeit", title="Zipfsches Gesetz im Schlagzeilen-Korpus")
ax.legend()
plt.show()

# Zähle die Terme, die genau einmal vorkommen (Hapax legomena)
hapax = sum(1 for c in counts.values() if c == 1)
print(f"{len(counts):,} verschiedene Terme, davon {hapax:,} ({hapax / len(counts):.0%}) genau einmal")

Daraus folgen zwei praktische Konsequenzen, die jedes weitere Skript prägen:

- **Der Großteil des Vokabulars ist selten.** Mehr als die Hälfte aller verschiedenen Terme sind *Hapax legomena* - Wörter, die genau einmal vorkommen. Sie liefern wenig statistische Evidenz, weshalb spätere Skripte oft Terme unter einer minimalen Dokumenthäufigkeit verwerfen.
- **Wenige Wörter leisten den Großteil der Arbeit.** Der Kopf der Verteilung ist eine kurze Liste sehr häufiger Terme, weshalb die Entfernung von Stoppwörtern die Zählungen so stark verändert, obwohl dabei so wenige verschiedene Wörter wegfallen.

Auch die Abflachung ganz oben ist üblich: Die Entfernung der Stoppwörter hat den Kopf abgeschnitten, also genau den Bereich, in dem der steilste Teil der Kurve läge.

## N-Gramme

Ein **N-Gramm** ist eine Folge von *n* benachbarten Tokens. Bigramme stellen Namen und feste Wendungen wieder her, die einzelne Tokens auseinanderreißen.

In [ ]:
# Zähle Paare benachbarter Tokens
def bigram_counts(frame: pd.DataFrame) -> Counter:
    return Counter(
        f"{a} {b}" for tokens in frame["tokens"] for a, b in zip(tokens, tokens[1:])
    )


# Die zehn häufigsten Bigramme pro Zeitung, mit Häufigkeit in Klammern
pd.DataFrame(
    {
        outlet: [f"{bg} ({n})" for bg, n in bigram_counts(group).most_common(10)]
        for outlet, group in corpus.groupby("outlet")
    }
)

Die Bigramme werden hier aus der *gefilterten* Tokenliste gebildet, sodass zwei Wörter, die durch ein Stoppwort getrennt waren, nebeneinander rücken. Für thematische Wendungen ist das oft erwünscht, es kann aber auch Paare erzeugen, die im Original nie nebeneinander standen. Das solltest du beim Lesen der Tabelle oben im Hinterkopf behalten. Die Tabelle zeigt außerdem die Namen, die Unigramme zerreißen: Personen (*Van der Bellen*, *Donald Trump*), Vereine, Ereignisse (*Panama Papers*). Und sie zeigt, was die Entscheidungen aus dem ersten Abschnitt übersehen haben. Ganz oben bei Der Standard steht `tv radiotipps`: das tägliche Fernsehprogramm, dessen Titel jeweils ein anderes Datum trägt (*TV- und Radiotipps für Sonntag, 9. August*) und das deshalb nicht als wiederkehrender Titel erkannt wurde. Auch `ende reise` bei der SZ ist eine Rubrik (*Ende der Reise*). Eine Häufigkeitstabelle ist oft der schnellste Weg, solche Reste zu finden; ob man sie entfernt, ist die nächste Entscheidung.

### Eine Konvention deutscher Schlagzeilen: die Dachzeile (Kicker)

Deutsche Schlagzeilen beginnen oft mit einem kurzen Themenetikett vor der eigentlichen Aussage - der *Dachzeile* oder dem Kicker: *"Flüchtlinge: …"*, *"FC Bayern - …"*. Wie oft eine Zeitung das tut und mit welchem Trennzeichen, ist ein stilistischer Fingerabdruck, und es wirkt sich direkt auf das Text Mining aus, denn das Wort in der Dachzeile ist faktisch ein Kategorie-Label mitten im Text.

In [ ]:
# Extrahiere die Dachzeile: alles vor dem ersten Doppelpunkt bzw. vor dem ersten " - " am Titelanfang
corpus["kicker_colon"] = corpus["title"].str.extract(r"^([^:]{2,30}):\s", expand=False)
corpus["kicker_dash"] = corpus["title"].str.extract(r"^([^:]{2,30}?) - ", expand=False)

# Anteil der Titel mit Dachzeile und die häufigsten Dachzeilen pro Zeitung
summary = corpus.groupby("outlet")[["kicker_colon", "kicker_dash"]].agg(lambda s: s.notna().mean())
summary["häufigste Dachzeilen"] = corpus.groupby("outlet").apply(
    lambda g: ", ".join(g["kicker_colon"].fillna(g["kicker_dash"]).value_counts().head(6).index),
    include_groups=False,
)
summary.style.format({"kicker_colon": "{:.0%}", "kicker_dash": "{:.0%}"})

Beide Zeitungen verwenden Dachzeilen, aber verschieden: Der Standard trennt sie mit einem **Doppelpunkt**, die SZ online überwiegend mit einem **Gedankenstrich** (`Kicker - Schlagzeile`) und seltener mit einem Doppelpunkt. Die häufigsten Dachzeilen sind Länder, Themen, Vereine - und Gattungsbezeichnungen wie *Kommentar*. Das solltest du dir merken: Jede Methode, die Zeitungen vergleicht, wird diese Etiketten aufgreifen, und ein Klassifikator, der Ressorts vorhersagen soll, liest die Antwort teilweise am ersten Wort ab.

## Das Problem der Komposita

Das Deutsche schreibt als ein Wort, was das Englische als Wortgruppe schreibt. Hier sind die längsten Tokens im Korpus:

In [ ]:
# Zeige die zehn längsten Tokens mit ihrer Länge
longest = sorted(counts, key=len, reverse=True)[:10]
for token in longest:
    print(f"{len(token):3}  {token}")

Jedes davon besteht aus mehreren zusammengeschweißten Konzepten. Für einen Zähler sind `flüchtlinge` und `flüchtlingskrise` unzusammenhängende Strings. Ein Thema, das in mehreren verschiedenen Komposita ausgedrückt wird, verteilt sich auf Terme, die jeweils zu selten sind, um aufzufallen.

Die einfachste Abhilfe ist, nach einem Bruchstück *innerhalb* der Tokens zu suchen:

In [ ]:
# Alle Tokens, die ein Bruchstück enthalten, nach Häufigkeit sortiert
def compounds_containing(fragment: str) -> pd.Series:
    return pd.Series({t: c for t, c in counts.items() if fragment in t}).sort_values(ascending=False)


fluecht = compounds_containing("flücht")
print(f"exaktes Token 'flüchtlinge': {counts['flüchtlinge']}  |  alle Tokens, die 'flücht' enthalten: {fluecht.sum()} "
      f"in {len(fluecht)} Formen")
fluecht.head(15)

Das Thema des Jahres verteilt sich auf viele Formen und Komposita - *Flüchtlingskrise*, *Flüchtlingslager*, *Flüchtlingspolitik* -, und die exakte Zählung erfasst nur einen Teil davon. Die Suche nach Teilstrings ist grob - sie findet auch *flüchtig* und *flüchtiger*, die mit Flucht im Sinne von Migration nichts zu tun haben, und Tippfehler wie *flüchtlin* -, sie zeigt aber, wie stark ein Thema bei exakter Übereinstimmung unterzählt wird. Echte Kompositumzerlegung (`Flüchtlingsunterkunft` in `Flüchtling` und `Unterkunft` aufzuteilen) braucht ein Wörterbuch oder ein statistisches Zerlegungsverfahren. Skript 05 umgeht das Problem mit **Zeichen-N-Grammen**, die in Wörter hineinsehen, ohne sie zu zerlegen.

## Zusammenfassung

| Schritt | Was er behebt | Was er kostet |
|---|---|---|
| gemeinsamer Zeitraum, nur Journalismus | Einzelstücke und Service-Seiten, die nicht zur Berichterstattung gehören | nichts |
| Platzhalter, Formate und Doubletten entfernen | Titel, die keine Meldung benennen, und Kopien, die Zählungen aufblähen | einige echte Artikel ohne brauchbaren Titel |
| HTML-Bereinigung, CSV-Spuren korrigieren | Markup und Export-Artefakte, die sich als Vokabular ausgeben | nichts |
| redaktioneller Teaser statt erster Absatz | Bildunterschriften im Text | nichts, wo es einen Teaser gibt |
| Tokenisierung mit regulärem Ausdruck | Satzzeichen, die Wörter zersplittern | Zahlen, sofern nicht gesondert behandelt |
| Casefolding | Großschreibung am Satzanfang, die Wörter aufspaltet | Unterscheidung von Nomen und Verb |
| Stoppwort-Entfernung | Funktionswörter, die die Zählungen dominieren | Negation, für alles, was mit Bedeutung zu tun hat |

Jeder dieser Schritte ist eine **Entscheidung**, keine Voreinstellung. Das nächste Skript ersetzt mehrere davon durch ein Sprachmodell, das die deutsche Grammatik kennt, und zeigt, welche der obigen Probleme es tatsächlich löst.

## Übungen

1. **Vokabular der Teaser.** Baue `tokens` aus der Spalte `text` statt aus `title` neu auf und erstelle die Tabelle der häufigsten Terme erneut. Welche Zeitung verändert sich am stärksten, und warum würdest du das nach dem Vergleich der Wortanzahlen im Abschnitt "Das Ergebnis" erwarten?
2. **Eine eigene Stoppwortliste.** Die NLTK-Liste wurde nicht für Schlagzeilen gemacht. Finde die zehn häufigsten verbleibenden Tokens über beide Zeitungen hinweg und entscheide für jedes, ob es ein Themenwort oder Schlagzeilen-Füllmaterial ist. Was würdest du ergänzen?
3. **Dachzeilen als Labels.** Entferne die Dachzeile (alles bis zum ersten Doppelpunkt oder `" - "`) aus den Titeln vor der Tokenisierung. Wie stark verändert sich die Liste der häufigsten Terme bei jeder Zeitung?
4. **Heapssches Gesetz.** Die Größe des Vokabulars wächst mit der Größe des Korpus, aber immer langsamer. Zeichne die Anzahl verschiedener Tokens gegen die Anzahl bisher gesehener Tokens, wobei du die Schlagzeilen von Der Standard in der Reihenfolge ihrer Veröffentlichung hinzufügst. Flacht die Kurve ab?
5. **Die Schwelle für Formate.** Setze `tm.FORMAT_MIN_REPEATS` auf 2 und auf 5 und lade das Korpus neu. Wie viele Artikel verändern sich, und welche Titel würdest du von Hand anders entscheiden?

## Quellen

* Schabus, D., Skowron, M., Trapp, M. (2017): *One Million Posts: A Data Set of German Online Discussions.* Proceedings of the 40th International ACM SIGIR Conference on Research and Development in Information Retrieval (SIGIR), 1241-1244. Daten: https://ofai.github.io/million-post-corpus/ (Lizenz: CC BY-NC-SA 4.0)
* Scialom, T., Dray, P.-A., Lamprier, S., Piwowarski, B., Staiano, J. (2020): *MLSUM: The Multilingual Summarization Corpus.* Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing (EMNLP), 8051-8067. Daten: https://github.com/ThomasScialom/MLSUM, hier über die Kopie im GEM-Benchmark auf https://huggingface.co/datasets/GEM/mlsum (Lizenz: nur für nicht-kommerzielle Forschung und Lehre)
* https://www.nltk.org/
* https://pandas.pydata.org/docs/
* https://matplotlib.org/
* https://de.wikipedia.org/wiki/Zipfsches_Gesetz